# UrbanPulse — Notebook 08: ECHO Stage B — Ecosystem State Machine

Bible §7 Stage B. Discovers which links causally influence which others (causal graph),
assigns each link a metabolic state every interval (timeline), switches causal direction
under backpressure (regime), and tracks cascade propagation (cascade events).

**Two Bible claims don't survive contact with the data** — see `DECISION_MAP.md` (B8).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import config, io_utils
from echo import ecosystem as eco
import pandas as pd, numpy as np

features = io_utils.load_parquet(config.FEATURES_PARQUET)
print(features.shape)

## 1. Bible's literal claim: raw cross-correlation

In [ ]:
pivot_raw = features.pivot_table(
    index=["day_number","minute_of_day"], columns="LINK_ID", values="mean_queue_s"
).sort_index()
s36, s16 = pivot_raw[36].to_numpy(), pivot_raw[16].to_numpy()

corr_36_16, lag_36_16 = eco.discover_causal_lag(s36, s16, max_lag=12)
corr_16_36, lag_16_36 = eco.discover_causal_lag(s16, s36, max_lag=12)
print(f"raw 36->16: corr={corr_36_16:.3f}  lag={lag_36_16*5}min")
print(f"raw 16->36: corr={corr_16_36:.3f}  lag={lag_16_36*5}min")
print("Direction signal (raw):", round(abs(corr_36_16 - corr_16_36), 3), "(near-zero = coin flip)")

**Problems:** 8 min isn't on the 5-min grid. Both directions nearly equal — shared AM/PM
cycle inflates correlation symmetrically, erasing direction.

## 2. Fix: de-seasonalize, then correlate residual shocks

In [ ]:
resid = eco.deseasonalize(features)
pivot_resid = resid.pivot_table(
    index=["day_number","minute_of_day"], columns="LINK_ID", values="queue_residual"
).sort_index()
r36, r16 = pivot_resid[36].to_numpy(), pivot_resid[16].to_numpy()

rcorr_36_16, rlag_36_16 = eco.discover_causal_lag(r36, r16, max_lag=12)
rcorr_16_36, rlag_16_36 = eco.discover_causal_lag(r16, r36, max_lag=12)
print(f"residual 36->16: corr={rcorr_36_16:.3f}  lag={rlag_36_16*5}min")
print(f"residual 16->36: corr={rcorr_16_36:.3f}  lag={rlag_16_36*5}min")
print("Direction signal (residual):", round(abs(rcorr_36_16 - rcorr_16_36), 3))

## 3. Build the full causal propagation graph

In [ ]:
graph = eco.build_causal_graph(features)
print(f"Edges: {graph.number_of_edges()}")
eco.save_causal_graph(graph)

if graph.has_edge(36, 16):
    d = graph[36][16]
    print(f"36->16 confirmed: corr={d['correlation_strength']:.3f}  lag={d['lag_minutes']}min")
else:
    print("36->16 NOT in graph at this threshold")

import pandas as pd
edges_df = pd.DataFrame([
    {"source": u, "target": v, **d} for u,v,d in graph.edges(data=True)
]).sort_values("correlation_strength", ascending=False)
edges_df.head(10)

## 4. Metabolic state timeline + regime switch

In [ ]:
timeline = eco.link_state_timeline(features)
print(timeline["state"].value_counts())
print(f"\nBackpressure regime: {(timeline['regime']=='backpressure').mean():.1%} of link-intervals")
timeline.head()

## 5. Cascade propagation tracker

In [ ]:
events = eco.cascade_events(graph, timeline)
print(f"Cascade events: {len(events)}  (Day 1: {(events['day_number']==1).sum()})")
if len(events):
    val_rate = events['n_validated'].sum() / events['n_downstream'].sum()
    print(f"Downstream validation rate: {val_rate:.1%}")
config.CASCADE_EVENTS_CSV.parent.mkdir(parents=True, exist_ok=True)
events.to_csv(config.CASCADE_EVENTS_CSV, index=False)
events.sort_values("n_downstream", ascending=False).head(10)

## 6. Full run + B8 gate

In [ ]:
out = eco.run()
print(f"edges: {out['n_edges']}")
print(f"36->16: {out['edge_36_16']}")
print(f"cascade events: {out['n_cascade_events']} (Day 1: {out['n_cascade_events_day1']})")
print(f"validation rate: {out['validation_rate']:.1%}")
passed = out['edge_36_16'] is not None and out['n_cascade_events_day1'] > 0
print(f"\nGATE: {'PASS' if passed else 'CHECK'}")